# EthicaAI — Melting Pot 20-Seed Benchmark
Install deps → Run experiment → Copy JSON output

In [ ]:
!pip install dmlab2d dm-meltingpot -q
import meltingpot
print('OK:', meltingpot.__version__)

In [ ]:
import numpy as np, json, sys, datetime
from meltingpot import substrate

N_SEEDS, N_STEPS = 20, 500

def run_ep(sub, pol_fn, seed=0):
    rng = np.random.RandomState(seed)
    env = substrate.build(substrate.get_config(sub))
    ts = env.reset()
    n = len(ts.observation)
    r = np.zeros(n)
    for t in range(N_STEPS):
        acts = {i: pol_fn(ts.observation[i], i, rng, t) for i in range(n)}
        ts = env.step(acts)
        for i in range(n): r[i] += ts.reward[i]
        if ts.last(): break
    env.close()
    return float(np.mean(r))

selfish = lambda o,i,rng,t: rng.randint(0,8)
commit = lambda o,i,rng,t: rng.choice([5,6,7]) if rng.random()<0.7 else rng.choice([0,1,2,3,4])

results = {}
for sub in ['commons_harvest__open']:
    print(f'=== {sub} ===')
    sr = {}
    for nm, fn in [('Selfish', selfish), ('Commitment', commit)]:
        print(f'  [{nm}]', end=' ')
        rews = []
        for s in range(N_SEEDS):
            rews.append(run_ep(sub, fn, s))
            if (s+1)%5==0: print(f's{s+1}={rews[-1]:.1f}', end=' ')
        sr[nm] = {'mean': float(np.mean(rews)), 'std': float(np.std(rews)), 'all': rews}
        print(f'=> {sr[nm]["mean"]:.2f}+-{sr[nm]["std"]:.2f}')
    results[sub] = sr

results['metadata'] = {'n_seeds': N_SEEDS, 'n_steps': N_STEPS,
    'platform': f'Colab Py{sys.version_info.major}.{sys.version_info.minor}',
    'date': datetime.date.today().isoformat()}
print('\n=== FINAL JSON (copy this) ===')
print(json.dumps(results, indent=2))